In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import requests
import re
from collections import Counter

In [12]:
df = pd.read_csv("hnaicker_project_summary.csv", sep=';')
df.columns = ['project','commit','author','time','message']
df['Month'] = pd.to_datetime(df['time'], unit='s').dt.strftime('%Y-%m')
df

,project,commit,author,time,message,Month
0,tyiannak_pyaudioanalysis,00204ff33b60af2b6da7b3f0ff82865fdcb971d1,Theodoros Giannakopoulos <tyiannak@gmail.com>,1432738598,movie added\n,2015-05
1,tyiannak_pyaudioanalysis,0030b718e92eb6b4798f7fe0bef8b9757aad54e3,Theodoros Giannakopoulos <tyiannak@gmail.com>,1550433437,Merge pull request #216 from cclauss/patch-2\n...,2019-02
2,tyiannak_pyaudioanalysis,0046c875f107159789cafd14cf07d04f3ac195c4,Theodoros Giannakopoulos <tyiannak@gmail.com>,1427731826,minor bug in spectrogram\n,2015-03
3,tyiannak_pyaudioanalysis,0082170c5623bbcb7aa4cf5be46d82d67bc23943,tyiannak <tyiannak@gmail.com>,1497539954,"bug fixes (NaN feature values, I/O problems wh...",2017-06
4,tyiannak_pyaudioanalysis,00f5979fd0339b74a70d83ec17bbe4c12c07501f,BStoimenov <B.L.Stoimenov@gmail.com>,1610517206,Merge branch 'dev' into master\n,2021-01
...,...,...,...,...,...,...
21799,dpiparo_vdt,fdb34ff87169659f67cd5c869b3dc21eb19767d1,dpiparo <dpiparo@4525493e-7705-40b1-a816-d608a...,1348413865,o Added stable tag to the repository\n,2012-09
21800,dpiparo_vdt,fddf8aebb3269db1d8a3109eaaa3b1c88de382d7,Vincenzo Innocente <vincenzo.innocente@cern.ch>,1491221955,add exp\n,2017-04
21801,dpiparo_vdt,fdea1cfcea097a09ddc362ab1ff21b10899fbad9,dpiparo <dpiparo@4525493e-7705-40b1-a816-d608a...,1363692328,o New tag targeting ARM archs\n,2013-03
21802,dpiparo_vdt,fe4efdcdaa8663193132cca1120ec55b12e4702e,Vincenzo Innocente <vincenzo.innocente@cern.ch>,1516717538,add inline to ease cuda porting\n,2018-01


In [ ]:
# Projects assigned
projects = [
    'tyiannak/pyAudioAnalysis',
    'FZJ-INM1-BDA/siibra-explorer',
    'spacetelescope/jwql',
    'smhyatt/deforest-watch-with-BFAST-in-CUDA',
    'eurostat/statistics-coded',
    'tnagler/VineCopula',
    'deeptime-ml/deeptime',
    'aiidateam/aiida-tutorials',
    'lepy/phuzzy',
    'dpiparo/vdt'
]

In [ ]:
#Part 2 data visualization

for project in projects:

    # Convert GitHub repository name to WoC V2412 project name
    prj = project.lower().replace('/', '_', 2)
    print("Processing:", project, "->", prj)
    df1 = df[df['project'] == prj]

    # Skip project if there is no data
    if df1.empty:
        print("No data found for:", prj)
        continue

    df2 = df1.groupby('Month').size().reset_index(name='Commits')
    monthly_counts = df2.copy()
    monthly_counts['time'] = pd.to_datetime(monthly_counts['Month'],errors='coerce')
    monthly_counts = monthly_counts.dropna(subset=['time'])
    if monthly_counts.empty:
        print("No valid dates found for:", prj)
        continue

    full_date_range = pd.DataFrame({
        'time': pd.date_range(start=monthly_counts['time'].min(),end=monthly_counts['time'].max(),freq='MS')})

    monthly_counts = pd.merge(full_date_range,monthly_counts,on='time',how='left')
    monthly_counts['Commits'] = monthly_counts['Commits'].fillna(0)
    monthly_counts['Month'] = monthly_counts['time'].dt.strftime('%Y-%m')

    # Create bar plot
    plt.figure(figsize=(10, 5))

    plt.bar(
        monthly_counts['Month'],
        monthly_counts['Commits']
    )

    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title("Commit Activity: " + project)

    plt.grid(axis="y")
    plt.tight_layout()

    ax = plt.gca()

    # Show every other x-axis label
    for label in ax.xaxis.get_ticklabels()[::2]:
        label.set_visible(False)

    plt.xticks(rotation=45, fontsize=7)

    # Safe filename
    filename = "hnaicker_" + prj + ".png"

    plt.savefig(filename, dpi=600)

    #plt.show()
    plt.close()

Processing: tyiannak/pyAudioAnalysis -> tyiannak_pyaudioanalysis
Processing: FZJ-INM1-BDA/siibra-explorer -> fzj-inm1-bda_siibra-explorer
Processing: spacetelescope/jwql -> spacetelescope_jwql
Processing: smhyatt/deforest-watch-with-BFAST-in-CUDA -> smhyatt_deforest-watch-with-bfast-in-cuda
Processing: eurostat/statistics-coded -> eurostat_statistics-coded
Processing: tnagler/VineCopula -> tnagler_vinecopula
Processing: deeptime-ml/deeptime -> deeptime-ml_deeptime
Processing: aiidateam/aiida-tutorials -> aiidateam_aiida-tutorials
Processing: lepy/phuzzy -> lepy_phuzzy
Processing: dpiparo/vdt -> dpiparo_vdt


In [32]:
#Commit Gaps

def woc_project_name(project):
    return project.lower().replace('/', '_', 2)


def get_github_info(project):
    url = f"https://api.github.com/repos/{project}"
    response = requests.get(url)
    if response.status_code != 200:
        print("GitHub API error for:", project)
        return {
            'stars': np.nan,
            'forks': np.nan,
            'last_commit_date': pd.NaT
        }
    repo_info = response.json()

    # Get most recent commit
    commits_url = f"https://api.github.com/repos/{project}/commits?per_page=1"
    commits_response = requests.get(commits_url)

    if commits_response.status_code == 200:
        commits = commits_response.json()

        if len(commits) > 0:
            last_commit_date = pd.to_datetime(
                commits[0]['commit']['committer']['date']
            )
        else:
            last_commit_date = pd.NaT
    else:
        last_commit_date = pd.NaT

    return {
        'stars': repo_info.get('stargazers_count', np.nan),
        'forks': repo_info.get('forks_count', np.nan),
        'last_commit_date': last_commit_date
    }



def classify_activity(monthly_counts):
    y = monthly_counts['Commits'].values
    if len(y) < 4:
        return 'irregular'
    midpoint = len(y) // 2
    first_half = np.mean(y[:midpoint])
    second_half = np.mean(y[midpoint:])
    x = np.arange(len(y))
    slope = np.polyfit(x, y, 1)[0]
    mean_commits = np.mean(y)

    if mean_commits == 0:
        return 'irregular'

    relative_slope = slope / mean_commits

    first_quarter = np.mean(y[:max(1, len(y)//4)])
    middle = np.mean(
        y[len(y)//3:2*len(y)//3]
    )
    last_quarter = np.mean(
        y[-max(1, len(y)//4):]
    )

    if (
        first_quarter > middle * 1.5
        and last_quarter > middle * 1.5
    ):
        return 'U-shaped'


    if relative_slope > 0.05:
        return 'rising'
    if relative_slope < -0.05:
        return 'declining'

    if len(y) >= 12:

        # Compare average activity in alternating periods
        autocorr_12 = np.corrcoef(
            y[:-12],
            y[12:]
        )[0, 1]

        if not np.isnan(autocorr_12) and autocorr_12 > 0.5:
            return 'cyclical'
        
    coefficient_variation = (
        np.std(y) / mean_commits
    )

    if coefficient_variation < 0.5:
        return 'steady'

    return 'irregular'


project_stats = []


for project in projects:
    prj = woc_project_name(project)
    print("Processing:", project)
    df1 = df[df['project'] == prj].copy()

    if df1.empty:
        print("  No data found.")
        continue

    df1['time'] = pd.to_datetime(
        df1['Month'],
        errors='coerce'
    )

    df1 = df1.dropna(subset=['time'])

    monthly_counts = (
        df1.groupby('time')
        .size()
        .reset_index(name='Commits')
    )


    full_date_range = pd.DataFrame({
        'time': pd.date_range(
            start=monthly_counts['time'].min(),
            end=monthly_counts['time'].max(),
            freq='MS'
        )
    })

    monthly_counts = pd.merge(
        full_date_range,
        monthly_counts,
        on='time',
        how='left'
    )

    monthly_counts['Commits'] = (
        monthly_counts['Commits']
        .fillna(0)
        .astype(int)
    )

    monthly_counts['Month'] = (
        monthly_counts['time']
        .dt.strftime('%Y-%m')
    )


    number_of_commits = len(df1)

    if 'author' in df1.columns:
        number_of_authors = df1['author'].nunique()

    else:
        number_of_authors = np.nan
        print(
            "  WARNING: 'author' column not found; "
            "number of authors set to NaN."
        )

    min_time = monthly_counts['time'].min()
    max_time = monthly_counts['time'].max()


    zero = monthly_counts['Commits'].eq(0)

    gaps = []

    i = 0

    while i < len(monthly_counts):

        if zero.iloc[i]:

            start = i

            while (
                i + 1 < len(monthly_counts)
                and zero.iloc[i + 1]
            ):
                i += 1

            end = i

            gap_length = end - start + 1

            gaps.append({
                'start': monthly_counts.iloc[start]['time'],
                'end': monthly_counts.iloc[end]['time'],
                'length': gap_length
            })

        i += 1


    if len(gaps) > 0:

        longest_gap = max(
            gaps,
            key=lambda x: x['length']
        )

        longest_gap_start = longest_gap['start']
        longest_gap_end = longest_gap['end']
        longest_gap_length = longest_gap['length']

    else:

        longest_gap_start = pd.NaT
        longest_gap_end = pd.NaT
        longest_gap_length = 0


    number_of_gaps = sum(
        gap['length'] >= 3
        for gap in gaps
    )


    if pd.notna(longest_gap_end):

        commits_after_gap = df1[
            df1['time'] > longest_gap_end
        ].shape[0]

    else:

        commits_after_gap = 0


    activity_pattern = classify_activity(
        monthly_counts
    )

    github_info = get_github_info(project)


    project_stats.append({

        'project': project,

        'number of commits':
            number_of_commits,

        'number of authors':
            number_of_authors,

        'max time':
            max_time.strftime('%Y-%m'),

        'min time':
            min_time.strftime('%Y-%m'),

        'number of stars':
            github_info['stars'],

        'number of forks':
            github_info['forks'],

        'last commit date':
            github_info['last_commit_date'],

        'LongestGapStart':
            (
                longest_gap_start.strftime('%Y-%m')
                if pd.notna(longest_gap_start)
                else ''
            ),

        'LongestGapEnd':
            (
                longest_gap_end.strftime('%Y-%m')
                if pd.notna(longest_gap_end)
                else ''
            ),

        'LongestGapLength':
            longest_gap_length,

        'NumCommitsAfterLastGap':
            commits_after_gap,

        'NumberOfGapsInTimeline':
            number_of_gaps,

        'ActivityPattern':
            activity_pattern
    })



stats_df = pd.DataFrame(project_stats)

stats_df.to_csv(
    'hnaicker_project_stats.csv',
    index=False
)



Processing: tyiannak/pyAudioAnalysis
Processing: FZJ-INM1-BDA/siibra-explorer
Processing: spacetelescope/jwql
Processing: smhyatt/deforest-watch-with-BFAST-in-CUDA
Processing: eurostat/statistics-coded
Processing: tnagler/VineCopula
Processing: deeptime-ml/deeptime
Processing: aiidateam/aiida-tutorials
Processing: lepy/phuzzy
Processing: dpiparo/vdt


In [33]:
stats_df

,project,number of commits,number of authors,max time,min time,number of stars,number of forks,last commit date,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline,ActivityPattern
0,tyiannak/pyAudioAnalysis,1048,102,2025-08,2014-08,6266,1211,2025-08-04 16:44:35+00:00,2022-12,2023-08,9,25,4,irregular
1,FZJ-INM1-BDA/siibra-explorer,4790,18,2025-10,2017-07,31,6,2026-07-26 07:41:40+00:00,2024-05,2024-06,2,334,0,irregular
2,spacetelescope/jwql,8539,56,2025-11,2017-11,69,47,2026-09-29 17:47:44+00:00,,,0,0,0,irregular
3,smhyatt/deforest-watch-with-BFAST-in-CUDA,643,4,2025-03,2019-09,0,0,2025-03-18 17:41:59+00:00,2021-08,2025-02,43,1,3,declining
4,eurostat/statistics-coded,813,35,2025-03,2019-01,42,16,2025-03-20 13:04:12+00:00,2022-06,2024-08,27,7,6,irregular
5,tnagler/VineCopula,657,23,2025-08,2015-09,101,34,2026-08-28 17:08:01+00:00,2023-08,2024-01,6,26,10,irregular
6,deeptime-ml/deeptime,3142,55,2025-04,2015-04,891,89,2026-06-26 12:34:29+00:00,2018-04,2018-12,9,2325,2,irregular
7,aiidateam/aiida-tutorials,1398,59,2025-10,2018-05,25,37,2025-10-02 11:27:44+00:00,2024-10,2025-09,12,3,6,irregular
8,lepy/phuzzy,459,11,2025-12,2017-06,3,2,2026-07-17 14:00:57+00:00,2021-01,2025-02,50,8,3,irregular
9,dpiparo/vdt,315,13,2025-05,2012-08,61,18,2025-04-03 13:33:36+00:00,2019-03,2022-02,36,18,13,irregular


Interpretation:

tyiannak/pyAudioAnalysis:

There are 4 gaps in this repo with an irregular pattern. The gaps do not follow a strict pattern that corresponds to any cycle

FZJ-INM1-BDA/siibra-explorer:

The longest gap in this repository was only 2 months long, so by the definition of 3 months for a gap, there are zero gaps in this repo. The pattern for commits are also declining drastically over time though there are still consistent commits, just not to the same degree. 

spacetelescope/jwql:

There are literally no gaps in this repository, there is at least one upload every month. The density of commits does seem to be decreasing over time, however.

smhyatt/deforest-watch-with-BFAST-in-CUDA:

There are 3 gaps in this commit with a pattern of declining commits over the course of this analysis. This is because there were around 4 months where commits were made and that's it. So in terms of trends, it's definitely declined since there have not been any commits for years. 

eurostat/statistics-coded:

There are 6 gaps of more than 3 months in this repository. The pattern of committing is still irregular, but the commit frequency has decreased drastically to zero until very recently when a few more commits were pushed.

tnagler/VineCopula:

There are a total of 10 measured gaps in this data. The majority of commits were added near the beginning of the repository's life with a drastic decrease in commits. The number of commits until today have been minimal, but are still present. 

deeptime-ml/deeptime:

Uniquely, this data set actually begins to increase in commits a long time after it was created. It follows a more U-shaped progression, though it is very irregular in that regard. It does have consistent data uploads with only 2 measured gaps.

aiidateam/aiida-tutorials:

This repo has 6 gaps and has an irregular commit history. I can't really characterize its trend. 

lepy/phuzzy:

This repo has 3 long gaps with the majority of commits occuring close to the creation of the repo. There are 4 months recently where there were nominal commits, but the trend is decreasing with a long gap of no commits between creation and recent times. 

dpiparo/vdt:

This repo has 13 gaps as the commit frequency is low, but when there are commits in a month, there are usually a lot. The pattern seems irregular with little way to characterize it. 

In [ ]:
#Part 3.1


summary_df = df.copy()
summary_df = summary_df.iloc[:, :5].copy()

summary_df.columns = [
    'project',
    'commit',
    'author',
    'time',
    'message'
]


summary_df['time'] = pd.to_datetime(summary_df['time'],unit='s')
summary_df['Month'] = summary_df['time'].dt.strftime('%Y-%m')

stats_df = pd.read_csv("hnaicker_project_stats.csv",sep=",")
stats_df["project"] = (stats_df["project"].str.lower().str.replace("/", "_", regex=False)
#print(summary_df.head())
#print(summary_df.columns)
#print(stats_df.columns)


Index(['project', 'commit', 'author', 'time', 'message', 'Month'], dtype='str')
Index(['project', 'number of commits', 'number of authors', 'max time',
       'min time', 'number of stars', 'number of forks', 'last commit date',
       'LongestGapStart', 'LongestGapEnd', 'LongestGapLength',
       'NumCommitsAfterLastGap', 'NumberOfGapsInTimeline', 'ActivityPattern'],
      dtype='str')


In [66]:
gap_commits = []

for _, row in stats_df.iterrows():

    project = row["project"]

    # Get longest gap
    gap_start = pd.to_datetime(row["LongestGapStart"])
    gap_end = pd.to_datetime(row["LongestGapEnd"])

    # Get commits for this project
    project_commits = summary_df[
        summary_df["project"] == project
    ].copy()

    # Sort chronologically
    project_commits = project_commits.sort_values("time")

    # Last 10 commits before gap
    before = project_commits[
        project_commits["time"] < gap_start
    ].tail(10).copy()

    before.insert(0, "pre/post", "pre")

    # First 10 commits after gap
    after_start = gap_end + pd.offsets.MonthBegin(1)

    after = project_commits[
        project_commits["time"] >= after_start
    ].head(10).copy()

    after.insert(0, "pre/post", "post")

    gap_commits.append(before)
    gap_commits.append(after)


# Combine all projects
gap_commits_df = pd.concat(
    gap_commits,
    ignore_index=True
)

# Save output
gap_commits_df.to_csv(
    "hnaicker_project_gap_commits.csv",
    sep=";",
    index=False
)



In [67]:
gap_commits_df

,pre/post,project,commit,author,time,message,Month
0,pre,tyiannak_pyaudioanalysis,756adcc5d1e9b4c29419a9e373d6151cf1476474,Theodoros Giannakopoulos <tyiannak@gmail.com>,2022-04-19 21:25:45,minor fix in aggregated similarity\n,2022-04
1,pre,tyiannak_pyaudioanalysis,71c67d921aa0d059b57e95103d08aaf35c177efa,Theodoros Giannakopoulos <tyiannak@gmail.com>,2022-04-19 21:26:51,Merge branch 'master' of https://github.com/ty...,2022-04
2,pre,tyiannak_pyaudioanalysis,52c3de799fa4a6a05a22324a63ab03474ee3fb04,Theodoros Giannakopoulos <tyiannak@gmail.com>,2022-05-09 20:56:02,Update README.md,2022-05
3,pre,tyiannak_pyaudioanalysis,06cca91a73236cf70651b688faa73e822bd450c2,Tim Gates <tim.gates@iress.com>,2022-07-02 13:10:52,docs: Fix a few typos\n\nThere are small typos...,2022-07
4,pre,tyiannak_pyaudioanalysis,fa73d3fa91b828cbdf775de6957c10ee15d95bde,Francesco Manfredi <francescomanfrediwd@gmail....,2022-07-14 21:42:02,signal shape check in read_audio_generic\n,2022-07
...,...,...,...,...,...,...,...
154,post,dpiparo_vdt,fc8550d96c2746a5d3ed0b637cc9579959728b93,Stephan Lachnit <stephanlachnit@debian.org>,2022-08-22 11:23:32,Merge 88c3c8df8bf67f9db67e1ae74f6b6f393682b1d1...,2022-08
155,post,dpiparo_vdt,93b10b8d848afaef2e3f3fb61751472bbd68ec27,Stephan Lachnit <stephanlachnit@debian.org>,2022-08-22 11:25:17,CMake: add SOVERSION\n\nSigned-off-by: Stephan...,2022-08
156,post,dpiparo_vdt,a627bf5e42287688e933d2486feb10edbe304af4,Stephan Lachnit <stephanlachnit@debian.org>,2022-08-22 11:28:23,Merge 93b10b8d848afaef2e3f3fb61751472bbd68ec27...,2022-08
157,post,dpiparo_vdt,774f44723485e4afd392082bbcff73f2707c83a2,Ian Briggs <ianbriggsutah@gmail.com>,2023-07-10 22:14:18,Reorder expression of better accuracy,2023-07


In [76]:
gap_df = pd.read_csv(
    "hnaicker_project_gap_commits.csv",
    sep=";"
)

def get_commit_messages(project, period):
    
    commits = gap_df[
        (gap_df["project"] == project) &
        (gap_df["pre/post"] == period)
    ].copy()
    
    commits = commits.sort_values("time")
    '''
    for i, message in enumerate(commits["message"], 1):
        print(f"{i}. {message}")
    '''
    return commits

In [79]:
#gap_df

Used the get_commit_messages function on each repo to conduct analysis for part 3.1

In [88]:
pd.set_option("display.max_colwidth", None)

In [ ]:
get_commit_messages("lepy_phuzzy", "pre")['message']

131                                                                           fix np.float -> float\n
132                                                                                       fix plots\n
133                                                                              add shape Constant\n
134                                                                                       fix plots\n
135    Merge 2e2c730c41cee99420be832f1936ced54c80b030 into 9dbde96c8054dcf8989a1218dca82c91b7ebfde1\n
136                                                                                add TruncLogNorm\n
137                                                                                add TruncLogNorm\n
138    Merge 2e2c730c41cee99420be832f1936ced54c80b030 into 33736fca859226f1225c4308418462ce6833f76c\n
Name: message, dtype: str

tyiannak/pyAudioAnalysis:

Pre: 
Bug fixes:Merges

Post:
Merges:Dependancy Updates


FZJ-INM1-BDA/siibra-explorer:

Pre:
Features:Bug fixes

Post:
Bug fixes:Merges


spacetelescope/jwql:

There are no gaps in this commit history so no need for analysis

smhyatt/deforest-watch-with-BFAST-in-CUDA:

Pre:
Documentation updates:Dependancy updates

Post:
Deletion:0

eurostat/statistics-coded:

Pre:
Feature development:Documentation updates

Post:
Feature development:Documentation updates

tnagler/VineCopula:

Pre: 
Bug fixes:Feature development

Post: 
Bug fixes:Dependancy updates


deeptime-ml/deeptime:

Pre:
Feature development:Bug fixes

Post:
Feature development:Bug fixes

aiidateam/aiida-tutorials:

Pre:
Bug fixes:Feature development

Post:
Bug fixes:Dependancy Updates

lepy/phuzzy:

Pre:
Automated bot contributions:Bug fixes

Post:
Bug fixes:Feature development

dpiparo/vdt:

Pre:
Documentation updates:Bug fixes

Post:
Feature development:Bug fixes


In [ ]:
def norm(name):
    return name.lower().replace("/", "_", 1)

new_cols = ["BeforeThemes", "AfterThemes", "HypothesizedGapReason", "HasRecovered",
            "WhyRecovered", "WhoRecovered", "Currentstatus", "RecentThemes", "Notes"]

NA = "N/A active Project"

# project: (Before, After, GapReason, Recovered, WhyRecovered, WhoRecovered, CurrentStatus, RecentThemes, Notes)
gap_info = {
    "tyiannak/pyAudioAnalysis": (
        "Bug fixes:Merges", "Merges:Dependancy Updates",
        "Maintainer-driven work stopped after Nov 2022; issues #382 (Dec 2022) and #390 (Aug 2023) were opened during the gap with no fix.",
        "Yes",
        "External contributor Frankie Eder opened PR #391 relaxing pandas/plotly pins (2023-09-29); the maintainer merged it on 2023-10-22.",
        "New contributor, merged by the original maintainer (Giannakopoulos)",
        "Active", "", ""),
    "FZJ-INM1-BDA/siibra-explorer": (
        "Features:Bug fixes", "Bug fixes:Merges",
        NA, "N/A", "N/A", "N/A", "Active", "", ""),
    "spacetelescope/jwql": (
        "N/A", "N/A", NA, "N/A", "N/A", "N/A", "Active", "", ""),
    "smhyatt/deforest-watch-with-BFAST-in-CUDA": (
        "Documentation updates:Dependancy updates", "Deletion:0",
        "Pre-gap commits are README writing and separating final code from draft work, ending with 'clean.' in 2021-07; it reads like a finished deliverable that was never maintained.",
        "No",
        "The only post-gap commit (2025-03-18) deletes 'scraps-messy-folder': housekeeping, not renewed development.",
        "Same author (S. Hyatt / Sarah Hyatt)",
        "Inactive", "", ""),
    "eurostat/statistics-coded": (
        "Feature development:Documentation updates", "Feature development:Documentation updates",
        "Last pre-gap work (2022-05) was a notebook figure update and README edit, with no end-of-life notice; likely the maintainer moved on or was reassigned.",
        "Yes",
        "First post-gap commits (2024-09-10) change the README contact address, followed by file uploads in 2025-01 and 2025-03, suggesting a maintainer/contact handover.",
        "Different account (k41m4n) than the pre-gap author (Meszaros Matyas); same person under a new account cannot be ruled out",
        "Inactive", "", ""),
    "tnagler/VineCopula": (
        "Bug fixes:Feature development", "Bug fixes:Dependancy updates",
        "Pre-gap commits end with Release 2.5.0 (2023-07), so the package was stable with nothing pending.",
        "Yes",
        "Bug report #91 (missing BB8 in BiCopName) led to fixes in 2024-02/03, then CRAN-compliance fixes and a version bump in 2024-08.",
        "Same maintainer (Nagler); Thibault Vatter added RVineCDF in 2024-11",
        "Active", "", ""),
    "deeptime-ml/deeptime": (
        "Feature development:Bug fixes", "Feature development:Bug fixes",
        "Pre-gap commits (2018-03) are only repository scaffolding (pybind, versioneer, travis, devtools, README); the original developer did not continue.",
        "Yes",
        "In 2019-01/02 a new developer (Moritz Hoffmann) began real work: dataset API draft, C++ subproject structure, covariance tests.",
        "New people (Hoffmann) rather than the scaffold's author (Scherer)",
        "Active", "", ""),
    "aiidateam/aiida-tutorials": (
        "Bug fixes:Feature development", "Bug fixes:Dependancy Updates",
        "Last pre-gap work (2024-09) was CI/linkcheck debugging and a docs update for AiiDA v2.5.1; a tutorial repo mainly changes when AiiDA changes.",
        "Yes",
        "In 2025-10 an external contributor (Subindev D) fixed typos in setup.md and the aiidalab-launch command; minor activity, not a new development push.",
        "New external contributor, merged by a maintainer (Goscinski)",
        "Active", "", ""),
    "lepy/phuzzy": (
        "Automated bot contributions:Bug fixes", "Bug fixes:Feature development",
        "Pre-gap activity was dominated by pyup/Streamlit bot commits and Travis fixes (2020-11/12); the single maintainer stopped adding human-written work after that.",
        "Yes",
        "First post-gap commit (2025-03-20) is 'fix np.float -> float', apparently a NumPy compatibility fix, followed by plot fixes and new shapes (Constant, TruncLogNorm).",
        "Same maintainer (lepy)",
        "Active", "", ""),
    "dpiparo/vdt": (
        "Documentation updates:Bug fixes", "Feature development:Bug fixes",
        "Pre-gap work (2018-2019) was release notes plus macOS/python-path build fixes; the library is small and mostly finished, so activity follows external build problems.",
        "Yes",
        "In 2022-03 Axel Naumann changed CMake to find python3 (fixes newest macOS); in 2022-08 Debian packager Stephan Lachnit contributed install-dir and SOVERSION CMake changes.",
        "New external contributors (Naumann, Lachnit), merged by the original maintainer (Piparo)",
        "Active", "", ""),
}

lookup = {norm(k): v for k, v in gap_info.items()}

for i, col in enumerate(new_cols):
    stats_df[col] = stats_df["project"].apply(lambda p: lookup[norm(p)][i])

stats_df.to_csv("hnaicker_project_stats.csv", index=False)
#stats_df

,project,number of commits,number of authors,max time,min time,number of stars,number of forks,last commit date,LongestGapStart,LongestGapEnd,...,ActivityPattern,BeforeThemes,AfterThemes,HypothesizedGapReason,HasRecovered,WhyRecovered,WhoRecovered,Currentstatus,RecentThemes,Notes
0,tyiannak_pyaudioanalysis,1048,102,2025-08,2014-08,6266,1211,2025-08-04 16:44:35+00:00,2022-12,2023-08,...,irregular,Bug fixes:Merges,Merges:Dependancy Updates,Maintainer-driven work stopped after Nov 2022; issues #382 (Dec 2022) and #390 (Aug 2023) were opened during the gap with no fix.,Yes,External contributor Frankie Eder opened PR #391 relaxing pandas/plotly pins (2023-09-29); the maintainer merged it on 2023-10-22.,"New contributor, merged by the original maintainer (Giannakopoulos)",Active,,
1,fzj-inm1-bda_siibra-explorer,4790,18,2025-10,2017-07,31,6,2026-07-26 07:41:40+00:00,2024-05,2024-06,...,irregular,Features:Bug fixes,Bug fixes:Merges,N/A active Project,N/A,N/A,N/A,Active,,
2,spacetelescope_jwql,8539,56,2025-11,2017-11,69,47,2026-09-29 17:47:44+00:00,NaN,NaN,...,irregular,N/A,N/A,N/A active Project,N/A,N/A,N/A,Active,,
3,smhyatt_deforest-watch-with-bfast-in-cuda,643,4,2025-03,2019-09,0,0,2025-03-18 17:41:59+00:00,2021-08,2025-02,...,declining,Documentation updates:Dependancy updates,Deletion:0,"Pre-gap commits are README writing and separating final code from draft work, ending with 'clean.' in 2021-07; it reads like a finished deliverable that was never maintained.",No,"The only post-gap commit (2025-03-18) deletes 'scraps-messy-folder': housekeeping, not renewed development.",Same author (S. Hyatt / Sarah Hyatt),Inactive,,
4,eurostat_statistics-coded,813,35,2025-03,2019-01,42,16,2025-03-20 13:04:12+00:00,2022-06,2024-08,...,irregular,Feature development:Documentation updates,Feature development:Documentation updates,"Last pre-gap work (2022-05) was a notebook figure update and README edit, with no end-of-life notice; likely the maintainer moved on or was reassigned.",Yes,"First post-gap commits (2024-09-10) change the README contact address, followed by file uploads in 2025-01 and 2025-03, suggesting a maintainer/contact handover.",Different account (k41m4n) than the pre-gap author (Meszaros Matyas); same person under a new account cannot be ruled out,Inactive,,
5,tnagler_vinecopula,657,23,2025-08,2015-09,101,34,2026-08-28 17:08:01+00:00,2023-08,2024-01,...,irregular,Bug fixes:Feature development,Bug fixes:Dependancy updates,"Pre-gap commits end with Release 2.5.0 (2023-07), so the package was stable with nothing pending.",Yes,"Bug report #91 (missing BB8 in BiCopName) led to fixes in 2024-02/03, then CRAN-compliance fixes and a version bump in 2024-08.",Same maintainer (Nagler); Thibault Vatter added RVineCDF in 2024-11,Active,,
6,deeptime-ml_deeptime,3142,55,2025-04,2015-04,891,89,2026-06-26 12:34:29+00:00,2018-04,2018-12,...,irregular,Feature development:Bug fixes,Feature development:Bug fixes,"Pre-gap commits (2018-03) are only repository scaffolding (pybind, versioneer, travis, devtools, README); the original developer did not continue.",Yes,"In 2019-01/02 a new developer (Moritz Hoffmann) began real work: dataset API draft, C++ subproject structure, covariance tests.",New people (Hoffmann) rather than the scaffold's author (Scherer),Active,,
7,aiidateam_aiida-tutorials,1398,59,2025-10,2018-05,25,37,2025-10-02 11:27:44+00:00,2024-10,2025-09,...,irregular,Bug fixes:Feature development,Bug fixes:Dependancy Updates,Last pre-gap work (2024-09) was CI/linkcheck debugging and a docs update for AiiDA v2.5.1; a tutorial repo mainly changes when AiiDA changes.,Yes,"In 2025-10 an external contributor (Subindev D) fixed typos in setup.md and the aiidalab-launch command; minor activity, not a new development push.","New external contributor, merged by a maintainer (Goscinski)",Active,,
8,lepy_phuzzy,459,11,2025-12,2017-06,3,2,2026-07-17 14:00:57+00:00,2021-01,2025-02,...,irregular,Automated bot contributions:Bug fixes,Bug fixes:Feature develo